In [ ]:
import boto3
import sagemaker
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

sns.set_theme()

sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name

In [ ]:
%store -r
%store

In [ ]:
ingest_create_athena_db_passed = False

In [ ]:
# Restore project S3 paths created during ingestion
project_prefix = project_prefix
s3_raw_cats_prefix = s3_raw_cats_prefix
s3_raw_noncats_prefix = s3_raw_noncats_prefix
s3_processed_combined_prefix = s3_processed_combined_prefix
s3_processed_cats_prefix = s3_processed_cats_prefix

print("Cats:", s3_raw_cats_prefix)
print("Noncats:", s3_raw_noncats_prefix)
print("Processed/combined:", s3_processed_combined_prefix)
print("Processed/cats:", s3_processed_cats_prefix)

# Setting up Athena

In [ ]:
database_name = "cat_landmarking"
manifest_table = "image_manifest"
landmarks_table = "cat_annotations"

s3_staging_dir = f"s3://{bucket}/{project_prefix}/athena/staging/"

engine = create_engine(
    f"awsathena+rest://@athena.{region}.amazonaws.com:443/"
    f"{database_name}"
    f"?s3_staging_dir={s3_staging_dir}"
)

In [ ]:
# Clean up this project's database from previous runs.
# Scoped to database_name only; other databases in the account are left untouched.
statement = f"DROP DATABASE IF EXISTS {database_name} CASCADE"
pd.read_sql(statement, engine)

# Create Database

In [ ]:
statement = f"CREATE DATABASE IF NOT EXISTS {database_name}"
pd.read_sql(statement, engine)

In [ ]:
statement = "SHOW DATABASES"
pd.read_sql(statement, engine)

# Create Cat Annotations Table

In [ ]:
# Creating landmarks table
s3_landmarks_location = f"{s3_processed_cats_prefix}/annotations"

statement_landmarks_table_spec = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {database_name}.{landmarks_table} (

    remote_path        VARCHAR(1024),
    local_path         VARCHAR(1024),

    left_eye           ARRAY<INT>,
    right_eye          ARRAY<INT>,
    mouth              ARRAY<INT>,
    left_ear_1         ARRAY<INT>,
    left_ear_2         ARRAY<INT>,
    left_ear_3         ARRAY<INT>,
    right_ear_1        ARRAY<INT>,
    right_ear_2        ARRAY<INT>,
    right_ear_3        ARRAY<INT>,

    version            VARCHAR(64),
    split              VARCHAR(32),

    left_eye_x         BIGINT,
    left_eye_y         BIGINT,
    right_eye_x        BIGINT,
    right_eye_y        BIGINT,
    mouth_x            BIGINT,
    mouth_y            BIGINT,
    left_ear_1_x       BIGINT,
    left_ear_1_y       BIGINT,
    left_ear_2_x       BIGINT,
    left_ear_2_y       BIGINT,
    left_ear_3_x       BIGINT,
    left_ear_3_y       BIGINT,
    right_ear_1_x      BIGINT,
    right_ear_1_y      BIGINT,
    right_ear_2_x      BIGINT,
    right_ear_2_y      BIGINT,
    right_ear_3_x      BIGINT,
    right_ear_3_y      BIGINT,

    file_size          BIGINT,
    width              BIGINT,
    height             BIGINT,
    label              BIGINT
)
STORED AS PARQUET
LOCATION '{s3_landmarks_location}'
"""

pd.read_sql(statement_landmarks_table_spec, engine)

# Create Manifest Table

In [ ]:
# Create manifest table
s3_metadata_location = f"{s3_processed_combined_prefix}/metadata"

pd.read_sql(f"DROP TABLE IF EXISTS {database_name}.{manifest_table}",
            engine)

statement_manifest_table_spec = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {database_name}.{manifest_table} (
    remote_path    STRING,
	local_path     STRING,
	file_size      BIGINT,
	width          BIGINT,
	height         BIGINT,
	label          BIGINT,
	source         STRING,
	version        STRING,
	split          STRING,
	ingest_time    STRING
)
STORED AS PARQUET
LOCATION '{s3_metadata_location}'
"""

pd.read_sql(statement_manifest_table_spec, engine)

# Checking Ingestion

In [ ]:
# Show tables in database
pd.read_sql(f"SHOW TABLES FROM {database_name}", engine)

In [ ]:
# Inspect head of manifest table
pd.read_sql(f"SELECT * FROM {database_name}.{manifest_table} LIMIT 5", engine)

In [ ]:
# Inspect head of landmarks table
pd.read_sql(f"SELECT * FROM {database_name}.{landmarks_table} LIMIT 5", 
            engine)

In [ ]:
ingest_create_athena_db_passed = True

In [ ]:
%store ingest_create_athena_db_passed

# EDA

In [ ]:
# Image counts by source and label (1 = cat, 0 = non-cat)
statement_label_counts = f"""
SELECT source, label, COUNT(*) AS n
FROM {database_name}.{manifest_table}
GROUP BY source, label
ORDER BY source, label
"""
pd.read_sql(statement_label_counts, engine)

In [ ]:
# Classifier splits: original images only (augmented cat images are for the keypoint model)
statement_split_counts = f"""
SELECT split, label, COUNT(*) AS n
FROM {database_name}.{manifest_table}
WHERE version = 'raw'
GROUP BY split, label
ORDER BY split, label
"""

pd.read_sql(statement_split_counts, engine)

In [ ]:
# File size by source and label. All images were re-encoded at the same JPEG quality
# during ingestion, so remaining differences come from resolution and content.
statement_file_size = f"""
SELECT
  source,
  label,
  MIN(file_size) AS min_bytes,
  APPROX_PERCENTILE(file_size, 0.5) AS p50_bytes,
  APPROX_PERCENTILE(file_size, 0.95) AS p95_bytes,
  MAX(file_size) AS max_bytes
FROM {database_name}.{manifest_table}
GROUP BY source, label
ORDER BY source, label
"""

pd.read_sql(statement_file_size, engine)

In [ ]:
statement_dimension_data = f"""
SELECT
	width,
	height
FROM {database_name}.{manifest_table}
"""

dimension_data = pd.read_sql(statement_dimension_data, engine)
dimension_data.head()

fig, ax = plt.subplots(1, 2, figsize=(10, 4))

ax[0].hist(dimension_data["width"])
ax[0].set_title("Width Dist.")

ax[1].hist(dimension_data["height"])
ax[1].set_title("Height Dist.")

plt.tight_layout()
plt.show()

In [ ]:
%store database_name
%store manifest_table    
%store landmarks_table 
%store s3_staging_dir